# Reservoir level (research prototype)

A tank gains or loses a signed amount of water each step, but it can never overflow
its capacity C nor drop below empty. So the level after each step is the previous
level plus this step's change, held back to the range [0, C]: any excess spills and
any shortfall is capped at empty. Because of the two limits the level is not simply
the running total of the changes -- once the tank is full or empty, further changes
in the same direction are lost. The changes use a fixed deterministic fill so the
result is reproducible; we report the checksum (sum of all levels) as the oracle.
The C port must reproduce this checksum.

In [ ]:
N_TEST = 1000000

# tank capacity; the level is always held to [0, CAP]
CAP = 20000

# deterministic signed change per step in the integer range [-500, 500]; exact integer
# arithmetic, so a C port using unsigned 64-bit arithmetic reproduces every value
def change_of(i):
    return (i * 2654435761) % 1000003 % 1001 - 500

a = [change_of(i) for i in range(N_TEST)]

In [ ]:
# the level starts empty and each step is held back to [0, CAP] (inherently sequential
# as written): out[i] = min(CAP, max(0, out[i-1] + a[i])), with out[-1] taken as 0.
def reservoir(a):
    out = []
    level = 0
    for change in a:
        level += change
        if level < 0:
            level = 0
        elif level > CAP:
            level = CAP
        out.append(level)
    return out

out = reservoir(a)
checksum = 0
for v in out:
    checksum += v
print('checksum =', checksum)

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any language)
# must reproduce this checksum within tol. The levels are all integers in [0, CAP] and
# their sum stays well below 2**53, so the checksum is exact in double precision and any
# correct port -- serial or parallel, however it splits the work -- must match it to
# well under 1. The levels are handed over as the terms of the checksum so the
# extraction tool can confirm the sum reproduces the expected value and measure its
# (here zero) spread under reordering.
ORACLE = {'kernel': 'reservoir',
          'params': {'N': N_TEST, 'CAP': CAP},
          'expected': {'CHECKSUM': float(checksum)},
          'terms': {'CHECKSUM': [float(v) for v in out]},
          'tol': 0.5}